# 4.3: detección propia de QRS (Pan-Tompkins a 100 Hz)

**Qué hace.** Detecta los picos R directamente del ECG, sin usar las marcas `.qrs` de la base, que son automáticas (`sqrs125`) y no están auditadas. Etapas clásicas de Pan y Tompkins (1985): pasabanda 5–15 Hz ida y vuelta, derivada de 5 puntos, cuadrado, integración en 150 ms y umbrales adaptativos con búsqueda hacia atrás y descarte de ondas T. Los parámetros están en segundos y en Hz (`config.QRS_*`), así que valen a 100 Hz. No supone la polaridad del R: el cuadrado de la derivada es igual para un R invertido, y para ubicar el pico se mide la polaridad de cada registro.

**Antecedente (antes de fijar estas reglas).** En una primera corrida exploratoria sobre desarrollo, el detector quedaba ciego en a03 (Se 0,51) y a17 (Se 0,78): el umbral quedaba por encima de los QRS y no bajaba. Se agregó que los umbrales se vuelvan a aprender si la búsqueda hacia atrás falla. Ese cambio se hizo mirando datos de desarrollo y se declara acá. Los parámetros de Pan y Tompkins no se ajustaron con datos.

**Reglas (fijadas antes de correr).**
1. **Pregunta:** ¿el detector propio puede reemplazar a las `.qrs` en el pipeline?
2. **Validación contra las `.qrs` (descriptiva, no decide):** sensibilidad (Se) y valor predictivo positivo (VPP) con tolerancia de 100 ms, por registro y con todos los latidos juntos, en los 47 registros de desarrollo.
3. **Revisión manual:** como las `.qrs` tienen errores, una discrepancia no es necesariamente un error del detector. Se juntan todas las discrepancias de los 47 registros (falsos positivos y falsos negativos) y se sortean **50** con semilla 2026, **antes de mirarlas**. Cada una se clasifica viendo 10 s de ECG en: *error del detector*, *error de las `.qrs`* o *dudosa*. Se reporta la proporción de cada clase.
4. **Decisión (esquema v3 §7.4):** F con los picos del detector propio contra F con las `.qrs`, sobre los mismos 22.123 minutos de E3a, con 2.000 remuestreos por sujeto. **Se adopta el detector si es no inferior en las dos medidas** (AUC por minuto y acierto por registro; límite inferior del IC 95 % de la diferencia ≥ −0,02). Si no, se siguen usando las `.qrs` y el detector queda como validación.
5. **Minutos:** si con el detector propio algún minuto de los 22.123 queda sin F, se reporta cuántos y la comparación se hace sobre los minutos que tienen F con los dos.
6. **Control (no decide):** la métrica de E1 (coherencia entre el EDR por área y la respiración real) en los 8 registros con respiración, como en `07`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import RESULTS_DIR, SEED, SPLIT_CSV
from src.data import load_ecg, load_qrs
from src.qrs.pan_tompkins import detect_r_peaks, validate_against_reference

FIG = RESULTS_DIR / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8})
AZUL, NARANJA, GRIS, TINTA = "#2a78d6", "#eb6834", "#a3a29c", "#2b2b2b"
split = pd.read_csv(SPLIT_CSV).set_index("record")
DEV = list(split.index[split.set == "dev"])
N_REVISION = 50
print(len(DEV), "registros de desarrollo")

47 registros de desarrollo
